In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

In [34]:
import json
import os
import time
from typing import Sequence, Tuple
import numpy as np
import jax
import jax.numpy as jnp
from jaxued.environments.underspecified_env import EnvParams, EnvState, UnderspecifiedEnv
from jaxued.utils import compute_max_mean_returns_epcount
import optax
from flax import struct
from flax.training.train_state import TrainState as BaseTrainState
import flax.linen as nn
from flax.linen.initializers import constant, orthogonal
import distrax
import orbax.checkpoint as ocp
import wandb
from jaxued.environments.maze.env_editor import LocalKeyMazeEditorRotateSplitAct, Observation
from jaxued.environments.maze.env_editor_sokoban import LocalSokobanMazeEditor, LocalSokobanMazeEditorRotate, LocalSokobanMazeEditorRotateSplitAct
from jaxued.linen import ResetRNN
from jaxued.environments import Maze, MazeRenderer, ObservedMazeRenderer, LocalObservedMazeRenderer, SokobanMaze
from jaxued.environments.maze import Level, ObservedLevel, make_level_generator, make_level_sokoban_generator
from jaxued.wrappers import AutoReplayWrapper
import chex

import logging
import hydra
from omegaconf import DictConfig, OmegaConf
import matplotlib.pyplot as plt

logger = logging.getLogger(__name__)

In [4]:
from omegaconf import OmegaConf
from hydra import initialize, compose


def load_hydra_config(config_path, config_name):
    # Initialize the Hydra context
    with initialize(config_path=config_path):
        # Compose the configuration
        cfg = compose(config_name=config_name)
        return cfg
    
config_path = "config"  # path to the directory containing the config file
config_name = "main_obs_gen"  # name of the config file without the extension

config = load_hydra_config(config_path, config_name)

if config["num_env_steps"] is not None:
    config["num_updates"] = config["num_env_steps"] // (config["num_train_envs"] * config["num_steps"])

if config['mode'] == 'eval':
    os.environ['WANDB_MODE'] = 'disabled'
    

/tmp/ipykernel_63251/3239372705.py:7: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path=config_path):


In [5]:
env = SokobanMaze(max_height=13, max_width=13, agent_view_size=5, normalize_obs=True, min_boxes=1)
sample_random_level = make_level_sokoban_generator(env.max_height, env.max_width, 60, 20)
adv_env = LocalSokobanMazeEditorRotateSplitAct(env, random_z_dimensions=config['adv_random_z_dimension'], zero_out_random_z=config['adv_zero_out_random_z'], num_agents=1, agent_view_size=5)
adv_env_renderer = ObservedMazeRenderer(env, tile_size=8, render_boxes=True)
adv_env_renderer_w_agents = LocalObservedMazeRenderer(env, tile_size=8, render_boxes=True)
env_renderer = MazeRenderer(env, tile_size=8, render_boxes=True)
env = AutoReplayWrapper(env)
env_params = env.default_params
adv_env_params = adv_env.default_params

def sample_empty_level():
    w, h = env._env.max_width, env._env.max_height
    return ObservedLevel(
        wall_map=jnp.zeros((h, w), dtype=jnp.bool_),
        box_map=jnp.zeros((h, w), dtype=jnp.bool_),
        box_goal_map=jnp.zeros((h, w), dtype=jnp.bool_),
        max_boxes=jnp.array(0, dtype=jnp.uint8),
        observation_map=jnp.zeros((h, w), dtype=jnp.bool_),
        width=w,
        height=h,
        
        # These values don't matter, as the adversary overwrites them.
        goal_pos=jnp.array([0, 0], dtype=jnp.uint32),
        agent_pos=jnp.array([1, 1], dtype=jnp.uint32),
        agent_dir=jnp.array(0, dtype=jnp.uint8),
        goal_placed=jnp.array(False, dtype=jnp.bool_),
    )

def sample_random_init_level(rng):
    w, h = env._env.max_width, env._env.max_height
    observation_map = jnp.zeros((h, w), dtype=jnp.bool_)
    rng_pos, rng_dir = jax.random.split(rng)
    agent_pos = jax.random.randint(rng_pos, (2,), 0, jnp.array([h, w]), dtype=jnp.uint32)
    agent_dir = jax.random.randint(rng_dir, (), 0, 4, dtype=jnp.uint8)
    return ObservedLevel(
        wall_map=jnp.zeros((h, w), dtype=jnp.bool_),
        box_map=jnp.zeros((h, w), dtype=jnp.bool_),
        box_goal_map=jnp.zeros((h, w), dtype=jnp.bool_),
        max_boxes=jnp.array(0, dtype=jnp.uint8),
        observation_map=observation_map.at[agent_pos[1], agent_pos[0]].set(True),
        width=w,
        height=h,
        
        # These values don't matter, as the adversary overwrites them.
        goal_pos=jnp.array([0, 0], dtype=jnp.uint32),
        agent_pos=agent_pos,
        agent_dir=agent_dir,
        goal_placed=jnp.array(False, dtype=jnp.bool_),
    )

def sample_test_level(rng):
    w, h = env._env.max_width, env._env.max_height
    observation_map = jnp.zeros((h, w), dtype=jnp.bool_)
    rng_pos, rng_dir = jax.random.split(rng)
    agent_pos = jax.random.randint(rng_pos, (2,), 0, jnp.array([h, w]), dtype=jnp.uint32)
    agent_dir = jax.random.randint(rng_dir, (), 1, 2, dtype=jnp.uint8)
    return ObservedLevel(
        wall_map=jnp.zeros((h, w), dtype=jnp.bool_),
        box_map=jnp.zeros((h, w), dtype=jnp.bool_).at[agent_pos[1]+1, agent_pos[0]].set(True),
        box_goal_map=jnp.zeros((h, w), dtype=jnp.bool_),
        max_boxes=jnp.array(config['max_boxes'], dtype=jnp.uint8),
        observation_map=jnp.ones((h, w), dtype=jnp.bool_),
        width=w,
        height=h,
        
        # These values don't matter, as the adversary overwrites them.
        goal_pos=jnp.array([0, 0], dtype=jnp.uint32),
        agent_pos=agent_pos,
        agent_dir=agent_dir,
        goal_placed=jnp.array(False, dtype=jnp.bool_),
    )

rng = jax.random.PRNGKey(14532)
rng, _rng = jax.random.split(rng)

In [35]:
@struct.dataclass
class TrainState:
    update_count: int
    pro_train_state: BaseTrainState
    adv_train_state: BaseTrainState
    init_train_state: BaseTrainState

class LevelInit(nn.Module):
    n_classes: int

    @nn.compact
    def __call__(self):
        # logits: shape (n_classes,)
        logits = self.param('logits', lambda rng, shape: jnp.zeros(shape), (self.n_classes,))
        pi = distrax.Categorical(logits=logits)
        value = self.param('value', lambda rng, shape: jnp.zeros(shape), ())
        return pi, value
    
NO_KL = False
BOX_PROB = 0.1
LAST_BOX_PROB = 0.1
WALL_PROB = (1 - (2 + LAST_BOX_PROB)*BOX_PROB)/2

EMPTY_PROB_SEP = False
EMPTY_PROB = 0.7 - (2 + LAST_BOX_PROB)*BOX_PROB
class DoubleCategorical(distrax.Distribution):
    def __init__(self, logits_1, logits_2):
        self.pi_1 = distrax.Categorical(logits=logits_1)
        self.pi_2 = distrax.Categorical(logits=logits_2)

        target_probs = jnp.array([WALL_PROB, WALL_PROB, BOX_PROB, BOX_PROB, BOX_PROB*0.1])
        if EMPTY_PROB_SEP:
            target_probs = jnp.array([EMPTY_PROB, 0.3, BOX_PROB, BOX_PROB, BOX_PROB*0.1])
        self.target_pi = distrax.Categorical(probs=target_probs)

    def _sample_n(self, key, n):
        key_1, key_2 = jax.random.split(key)
        samples_1 = self.pi_1._sample_n(key_1, n)
        samples_2 = self.pi_2._sample_n(key_2, n)
        return jnp.stack((samples_1, samples_2), axis=-1)

    def log_prob(self, value):
        log_prob_1 = self.pi_1.log_prob(value[..., 0])
        log_prob_2 = self.pi_2.log_prob(value[..., 1])
        return log_prob_1 + log_prob_2

    def entropy(self):
        #return self.pi_1.entropy() + self.pi_2.entropy()
        if NO_KL:
            return self.pi_1.entropy()
        return self.pi_1.entropy() - self.pi_2.kl_divergence(self.target_pi)

    def event_shape(self):
        return (2,)

class ActorCritic(nn.Module):
    action_dim: Sequence[int]
    
    @nn.compact
    def __call__(self, inputs, hidden):
        obs, dones = inputs
        
        img_embed = nn.Conv(32, kernel_size=(3, 3), strides=(1, 1), padding="VALID")(obs.image)
        img_embed = img_embed.reshape(*img_embed.shape[:-3], -1)
        img_embed = nn.relu(img_embed)
        
        dir_embed = jax.nn.one_hot(obs.agent_dir, 4)
        dir_embed = nn.Dense(5, kernel_init=orthogonal(np.sqrt(2)), bias_init=constant(0.0), name="scalar_embed")(dir_embed)
        
        embedding = jnp.concatenate((img_embed, dir_embed, obs.has_key[..., None]), axis=-1)

        hidden, embedding = ResetRNN(nn.OptimizedLSTMCell(features=256))((embedding, dones), initial_carry=hidden)
        embedding = nn.LayerNorm()(embedding)

        actor_mean = nn.Dense(256, kernel_init=orthogonal(2), bias_init=constant(0.0), name="actor0")(embedding)
        actor_mean = nn.LayerNorm()(actor_mean)
        actor_mean = nn.tanh(actor_mean)
        actor_mean = nn.Dense(self.action_dim, kernel_init=orthogonal(0.01), bias_init=constant(0.0), name="actor1")(actor_mean)
        pi = distrax.Categorical(logits=actor_mean)

        critic = nn.Dense(256, kernel_init=orthogonal(2), bias_init=constant(0.0), name="critic0")(embedding)
        critic = nn.LayerNorm()(critic)
        critic = nn.tanh(critic)
        critic = nn.Dense(1, kernel_init=orthogonal(1.0), bias_init=constant(0.0), name="critic1")(critic)

        return hidden, pi, jnp.squeeze(critic, axis=-1)
    
    @staticmethod
    def initialize_carry(batch_dims):
        return nn.OptimizedLSTMCell(features=256).initialize_carry(jax.random.PRNGKey(0), (*batch_dims, 256))


class AdversaryActorCritic(nn.Module):
    # The adversary's network architecture
    action_dim: Sequence[int]
    max_timesteps: int = 50
    student_max_timesteps: int = 250
    max_boxes: int = 10
    
    @nn.compact
    def __call__(self, inputs: Tuple[Observation, chex.Array], hidden):
        obs, dones = inputs
        
        img_embed = nn.Conv(32, kernel_size=(3, 3), strides=(1, 1), padding="VALID")(jnp.concatenate((obs.image, obs.observation_map, obs.agent_boxes), axis=-1))
        img_embed = img_embed.reshape(*img_embed.shape[:-3], -1)
        img_embed = nn.relu(img_embed)
        
        time_value = nn.Embed(self.max_timesteps + 1, 10, name="time_embed", embedding_init=orthogonal(1.0))(jnp.clip(obs.time, None, self.max_timesteps))
        student_time_value = nn.Embed(self.student_max_timesteps + 1, 10, name="student_time_embed", embedding_init=orthogonal(1.0))(jnp.clip(obs.agent_steps, None, self.student_max_timesteps))
        dirs_embedding = jax.nn.one_hot(obs.agent_dirs, 4).reshape(*obs.agent_dirs.shape[:2], -1)
        box_embedding = nn.Embed(self.max_boxes + 1, 10, name="box_embed", embedding_init=orthogonal(1.0))(jnp.clip(obs.box_count, None, self.max_boxes))
        box_goal_embedding = nn.Embed(self.max_boxes + 1, 10, name="box_goal_embed", embedding_init=orthogonal(1.0))(jnp.clip(obs.box_goal_count, None, self.max_boxes))
        embedding = jnp.concatenate((img_embed, time_value, student_time_value, obs.agent_values, obs.place_goal[..., None], obs.goal_placed, dirs_embedding, box_embedding, box_goal_embedding), axis=-1)

        hidden, embedding = ResetRNN(nn.OptimizedLSTMCell(features=256))((embedding, dones), initial_carry=hidden)
        embedding = nn.LayerNorm()(embedding)

        actor_mean = nn.Dense(256, kernel_init=orthogonal(2), bias_init=constant(0.0), name="actor0")(embedding)
        actor_mean = nn.LayerNorm()(actor_mean)
        actor_mean = nn.tanh(actor_mean)
        actor_mean_0 = nn.Dense(25, kernel_init=orthogonal(0.01), bias_init=constant(0.0), name="actor10")(actor_mean)
        actor_mean_1 = nn.Dense(5, kernel_init=orthogonal(0.01), bias_init=constant(0.0), name="actor11")(actor_mean)

        # Mask out this
        actor_mean_0 = jnp.where(obs.action_mask[0], actor_mean_0, -jnp.inf)
        actor_mean_1 = jnp.where(obs.action_mask[1], actor_mean_1, -jnp.inf)
        pi = DoubleCategorical(logits_1=actor_mean_0, logits_2=actor_mean_1)

        critic = nn.Dense(256, kernel_init=orthogonal(2), bias_init=constant(0.0), name="critic0")(embedding)
        critic = nn.LayerNorm()(critic)
        critic = nn.tanh(critic)
        critic = nn.Dense(1, kernel_init=orthogonal(1.0), bias_init=constant(0.0), name="critic1")(critic)

        return hidden, pi, jnp.squeeze(critic, axis=-1)
    
    @staticmethod
    def initialize_carry(batch_dims):
        return nn.OptimizedLSTMCell(features=256).initialize_carry(jax.random.PRNGKey(0), (*batch_dims, 256))

In [ ]:
def create_train_state(rng):
    def create_inner_train_state(rng, env, env_params, network_cls, prefix, network_kws={}):
        def linear_schedule(count):
            frac = (
                1.0
                - (count // (config[f"{prefix}num_minibatches"] * config[f"{prefix}epoch_ppo"]))
                / config["num_updates"]
            )
            return config[f"{prefix}lr"] * frac
        if env != None:
            obs, _ = env.reset_to_level(rng, sample_empty_level(), env_params)
            obs = jax.tree_map(
                lambda x: jnp.repeat(jnp.repeat(x[None, ...], config["num_train_envs"], axis=0)[None, ...], 256, axis=0),
                obs,
            )
            init_x = (obs, jnp.zeros((256, config["num_train_envs"])))
            network = network_cls(env.action_space(env_params).n, **network_kws)
            network_params = network.init(rng, init_x, network_cls.initialize_carry((config["num_train_envs"],)))
        else:
            network = network_cls(**network_kws)
            network_params = network.init(rng)
        learning_rate = linear_schedule if config[f"{prefix}anneal_lr"] else config[f"{prefix}lr"]
        tx = optax.chain(
            optax.clip_by_global_norm(config[f"{prefix}max_grad_norm"]),
            #optax.adam(learning_rate=linear_schedule, eps=1e-5),
            #optax.adam(learning_rate=config[f"{prefix}lr"], eps=1e-5),
            optax.adam(learning_rate=learning_rate, eps=1e-5),
        )
        return BaseTrainState.create(
            apply_fn=network.apply,
            params=network_params,
            tx=tx,
        )
    rng_pro, rng_adv, rng_init = jax.random.split(rng, 3)
    return TrainState(
        update_count = 0,
        pro_train_state = create_inner_train_state(rng_pro, env, env_params, ActorCritic, "student_"),
        adv_train_state = create_inner_train_state(rng_adv, adv_env, adv_env_params, AdversaryActorCritic, "adv_", network_kws={"max_timesteps": config["adv_num_steps"], "student_max_timesteps": config["max_steps_in_episode"], "max_boxes": config["max_boxes"]}),
        init_train_state = create_inner_train_state(rng_init, None, None, LevelInit, "adv_", network_kws={"n_classes": config['max_boxes']}),
    )    

In [ ]:
def update_init_actor_critic(
    rng: chex.PRNGKey,
    train_state: TrainState,
    batch: chex.ArrayTree,
    num_envs: int,
    n_minibatch: int,
    n_epochs: int,
    clip_eps: float,
    entropy_coeff: float,
    critic_coeff: float,
    update_grad: bool=True,
) -> Tuple[Tuple[chex.PRNGKey, TrainState], chex.ArrayTree]:
    """This function takes in a rollout, and PPO hyperparameters, and updates the train state.

    Args:
        rng (chex.PRNGKey): 
        train_state (TrainState): 
        init_hstate (chex.ArrayTree): 
        batch (chex.ArrayTree): obs, actions, dones, log_probs, values, targets, advantages
        num_envs (int): 
        n_steps (int): 
        n_minibatch (int): 
        n_epochs (int): 
        clip_eps (float): 
        entropy_coeff (float): 
        critic_coeff (float): 
        update_grad (bool, optional): If False, the train state does not actually get updated. Defaults to True.

    Returns:
        Tuple[Tuple[chex.PRNGKey, TrainState], chex.ArrayTree]: It returns a new rng, the updated train_state, and the losses. The losses have structure (loss, (l_vf, l_clip, entropy))
    """
    actions, log_probs, values, targets, advantages = batch
    batch = actions, log_probs, values, targets, advantages
    
    def update_epoch(carry, _):
        def update_minibatch(train_state, minibatch):
            actions, log_probs, values, targets, advantages = minibatch
            
            def loss_fn(params):
                pi, values_pred = train_state.apply_fn(params)
                log_probs_pred = pi.log_prob(actions)
                entropy = pi.entropy().mean()

                ratio = jnp.exp(log_probs_pred - log_probs)
                A = (advantages - advantages.mean()) / (advantages.std() + 1e-5)
                l_clip = (-jnp.minimum(ratio * A, jnp.clip(ratio, 1 - clip_eps, 1 + clip_eps) * A)).mean()

                values_pred_clipped = values + (values_pred - values).clip(-clip_eps, clip_eps)
                l_vf = 0.5 * jnp.maximum((values_pred - targets) ** 2, (values_pred_clipped - targets) ** 2).mean()

                loss = l_clip + critic_coeff * l_vf - entropy_coeff * entropy

                return loss, (l_vf, l_clip, entropy)

            grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
            loss, grads = grad_fn(train_state.params)
            if update_grad:
                train_state = train_state.apply_gradients(grads=grads)
            return train_state, loss

        rng, train_state = carry
        rng, rng_perm = jax.random.split(rng)
        permutation = jax.random.permutation(rng_perm, num_envs)
        minibatches = jax.tree_map(
            lambda x: jnp.take(x, permutation, axis=1)
            .reshape(x.shape[0], n_minibatch, -1, *x.shape[2:])
            .swapaxes(0, 1),
            batch,
        )
        
        train_state, losses = jax.lax.scan(update_minibatch, train_state, minibatches)
        return (rng, train_state), losses

    return jax.lax.scan(update_epoch, (rng, train_state), None, n_epochs)

In [39]:
rng, _rng = jax.random.split(rng)
train_state = create_train_state(_rng)

In [133]:
i_t_s = train_state.init_train_state

pi, value = i_t_s.apply_fn(i_t_s.params)

rng, _rng = jax.random.split(rng)
actions = pi.sample(seed=_rng, sample_shape=(256,))
log_probs = pi.log_prob(actions)
values = jnp.ones_like(log_probs)*value

returns = (actions <= 2)*10.0
advantages = returns - values
targets = returns

rollout = actions, log_probs, values, targets, advantages
rollout = jax.tree_map(lambda x: x[None, ...], rollout)

In [134]:
(rng, init_train_state), init_losses = update_init_actor_critic(
    rng,
    train_state.init_train_state,
    rollout,
    config["num_train_envs"],
    config[f"adv_num_minibatches"],
    config[f"adv_epoch_ppo"],
    config[f"adv_clip_eps"],
    config[f"adv_entropy_coeff"],
    config[f"adv_critic_coeff"],
    update_grad=True,
)

train_state = train_state.replace(init_train_state=init_train_state)

In [135]:
new_pi, new_value = init_train_state.apply_fn(init_train_state.params)
new_pi.probs
new_value

Array(0.3092554, dtype=float32)